In [ ]:
import pandas as pd
import numpy as np
import copy

In [ ]:
# genes-pathways annotation

path = './kegg_pathway/kegg_hsa.gmt'

files = open(path,encoding='utf-8')

files = files.readlines()

paways_genes_dict = {}
for i in files: 
    paways_genes_dict[i.split('\t')[0].split('_')[0]] = i.replace('\n','').split('\t')[2:] 


In [ ]:
#mirna-pathways annotation
path = './kegg_pathway/kegg_anano.txt'

files = open(path,encoding='utf-8')

files = files.readlines()

paways_mirna_dict = {}
for i in files:
     keys = i.split(',')[0].split('|')[1]
     values1 = i.split(',')[1:-1]
     values2 =  i.split(',')[-1].replace('\n','')
     values1.append(values2)
     values1 =list(set(values1)) 
     paways_mirna_dict[keys] = values1

In [ ]:
union_kegg = list(set(paways_genes_dict.keys()).intersection(set(paways_mirna_dict.keys())))

In [ ]:
paways_genes_dicts ={}
paways_mirna_dicts ={}

for i in union_kegg:
    paways_genes_dicts[i] = paways_genes_dict[i]
    
for i in union_kegg:
    paways_mirna_dicts[i] = paways_mirna_dict[i]    

In [ ]:

genes_existed_pathway = []

mirna_existed_pathway = []

for index in paways_genes_dicts.keys():
    genes_existed_pathway = genes_existed_pathway+ list(paways_genes_dicts[index])
genes_existed_pathway = set(genes_existed_pathway)


for index in paways_mirna_dicts.keys():
    mirna_existed_pathway = mirna_existed_pathway+ list(paways_mirna_dicts[index])
mirna_existed_pathway = set(mirna_existed_pathway)



In [ ]:
print(len(genes_existed_pathway))
print(len(mirna_existed_pathway))

In [ ]:
#Loading data

miRNA_data = pd.read_csv("./aml_data/miRNA_data.csv",index_col = 0)
mRNA_data = pd.read_csv("./aml_data/mRNA_data.csv",index_col = 0)
example_case = pd.read_csv('./aml_data/response.csv',index_col=0)

In [ ]:
print(mRNA_data.shape)
print(miRNA_data.shape)

In [ ]:
# mRNA_data

In [ ]:
# example_case

In [ ]:
union_gene_miRNA = list(miRNA_data.columns)
union_gene_mRNA = list(mRNA_data.columns)

In [ ]:
pathway_union = list(paways_genes_dicts.keys())
len(pathway_union)

In [ ]:
mask_list = [union_gene_mRNA]

gene_pathway_bp_dfs = []


for i in range(len(mask_list)):
    pathways_genes = np.zeros((len(pathway_union), len(mask_list[i]))) 
    for p  in pathway_union:
        gs = paways_genes_dicts[p]
        g_inds = [mask_list[i].index(g) for g in gs if g in mask_list[i]]
        p_ind = pathway_union.index(p)
        pathways_genes[p_ind, g_inds] = 1
    gene_pathway_bp = pd.DataFrame(pathways_genes, index=pathway_union, columns=mask_list[i])
    
    gene_pathway_bp_dfs.append(gene_pathway_bp)
    

pathways_genes = np.zeros((len(pathway_union), len(union_gene_miRNA))) 
for p  in pathway_union:
    gs = paways_mirna_dicts[p]
    g_inds = [union_gene_miRNA.index(g) for g in gs if g in union_gene_miRNA]
    p_ind = pathway_union.index(p)
    pathways_genes[p_ind, g_inds] = 1
gene_pathway_bp = pd.DataFrame(pathways_genes, index=pathway_union, columns=union_gene_miRNA)

gene_pathway_bp_dfs.append(gene_pathway_bp)
    


In [ ]:
# len(gene_pathway_bp_dfs)

In [ ]:
# gene_pathway_bp_dfs

In [ ]:
import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers
import tensorflow as tf
from tensorflow.keras.initializers import glorot_uniform, Initializer
from tensorflow.keras.layers import Input, Dense, Dropout, Embedding, GlobalAveragePooling1D,Layer
from tensorflow.keras import initializers,activations,regularizers
from tensorflow.keras.regularizers import Regularizer
from tensorflow.keras.models import Model
from tensorflow.keras.optimizers import Adam
from tensorflow.keras.losses import SparseCategoricalCrossentropy
from tensorflow.keras import backend as K
from tensorflow.keras.regularizers import l2
from tensorflow.python.framework.ops import disable_eager_execution
 
tf.compat.v1.disable_eager_execution()

In [ ]:
class Biological_module(Layer):
    def __init__(self, units, mapp=None, nonzero_ind=None, kernel_initializer='glorot_uniform', W_regularizer=None,
                 activation='tanh', use_bias=True,bias_initializer='zeros', bias_regularizer=None,
                 bias_constraint=None,**kwargs):
        
        self.units = units
        self.activation = activation
        self.mapp = mapp
        self.nonzero_ind = nonzero_ind
        self.use_bias = use_bias
        
        self.kernel_initializer = initializers.get(kernel_initializer)
        self.kernel_regularizer = regularizers.get(W_regularizer)
        self.bias_initializer = initializers.get(bias_initializer)
        self.bias_regularizer = regularizers.get(bias_regularizer)
        self.activation_fn = activations.get(activation)
        super(Biological_module, self).__init__(**kwargs)

        
    def build(self, input_shape):
        
        input_dim = input_shape[1]
   

        if not self.mapp is None:
            self.mapp = self.mapp.astype(np.float32)

   
        if self.nonzero_ind is None:
            nonzero_ind = np.array(np.nonzero(self.mapp)).T
            self.nonzero_ind = nonzero_ind

        self.kernel_shape = (input_dim, self.units)
        

        nonzero_count = self.nonzero_ind.shape[0]   


        self.kernel_vector = self.add_weight(name='kernel_vector',
                                             shape=(nonzero_count,),
                                             initializer=self.kernel_initializer,
                                             regularizer=self.kernel_regularizer,
                                             trainable=True)
        if self.use_bias:
            self.bias = self.add_weight(shape=(self.units,),
                                        initializer=self.bias_initializer,
                                        name='bias',
                                        regularizer=self.bias_regularizer
                                        )
        else:
            self.bias = None

        super(Biological_module, self).build(input_shape)  
      

    def call(self, inputs):
        
        
        trans = tf.scatter_nd(tf.constant(self.nonzero_ind, tf.int32), self.kernel_vector,
                           tf.constant(list(self.kernel_shape)))
    
        output = K.dot(inputs, trans)
        
    
        if self.use_bias:
            output = K.bias_add(output, self.bias)
            
        if self.activation_fn is not None:
            output = self.activation_fn(output)

        return output

    def get_config(self):
        config = {
            'units': self.units,
            'activation': self.activation,
            'use_bias': self.use_bias,
            'nonzero_ind': np.array(self.nonzero_ind),
          
            'bias_initializer': initializers.serialize(self.bias_initializer),
            'bias_regularizer': regularizers.serialize(self.bias_regularizer),

            'kernel_initializer': initializers.serialize(self.kernel_initializer),
            'W_regularizer': regularizers.serialize(self.kernel_regularizer),

        }
        base_config = super(Biological_module, self).get_config()
        return dict(list(base_config.items()) + list(config.items()))

    def compute_output_shape(self, input_shape):
      
        return (input_shape[0], self.units)

In [ ]:
#Multiple Attention Module,Learning inter-sample correlations through multiple attention mechanisms
class MultiHeadSelfAttention(Layer):
    def __init__(self, output_dim, num_heads,dropout_rate=0.1, W_regularizer=None, **kwargs):
        self.output_dim = output_dim
        self.num_heads = num_heads
        self.dropout_rate = dropout_rate
        self.kernel_regularizer = regularizers.get(W_regularizer)
        super(MultiHeadSelfAttention, self).__init__(**kwargs)

    def build(self, input_shape):
        assert self.output_dim % self.num_heads == 0, "Output dimension must be divisible by the number of heads."
        self.depth = self.output_dim // self.num_heads

        self.query_kernel = self.add_weight(name='query_kernel',
                                            shape=(input_shape[-1], self.output_dim),
                                            initializer='uniform',
                                            regularizer=self.kernel_regularizer,
                                            trainable=True)
        self.key_kernel = self.add_weight(name='key_kernel',
                                          shape=(input_shape[-1], self.output_dim),
                                          initializer='uniform',
                                          regularizer=self.kernel_regularizer,
                                          trainable=True)
        self.value_kernel = self.add_weight(name='value_kernel',
                                            shape=(input_shape[-1], self.output_dim),
                                            initializer='uniform',
                                            regularizer=self.kernel_regularizer,
                                            trainable=True)
        self.dropout = Dropout(self.dropout_rate)
        super(MultiHeadSelfAttention, self).build(input_shape)

    def call(self, x,training=False):
        batch_size = tf.shape(x)[0]

        # Linear projections
        Q = tf.tensordot(x, self.query_kernel, axes=[-1, 0])  # (batch_size, seq_len, output_dim)
        K = tf.tensordot(x, self.key_kernel, axes=[-1, 0])    # (batch_size, seq_len, output_dim)
        V = tf.tensordot(x, self.value_kernel, axes=[-1, 0])  # (batch_size, seq_len, output_dim)

        # Reshape to (batch_size, seq_len, num_heads, depth)
        Q = tf.reshape(Q, (batch_size, -1, self.num_heads, self.depth))
        K = tf.reshape(K, (batch_size, -1, self.num_heads, self.depth))
        V = tf.reshape(V, (batch_size, -1, self.num_heads, self.depth))

        # Transpose to (batch_size, num_heads, seq_len, depth)
        Q = tf.transpose(Q, perm=[0, 2, 1, 3])
        K = tf.transpose(K, perm=[0, 2, 1, 3])
        V = tf.transpose(V, perm=[0, 2, 1, 3])

        # Scaled dot-product attention
        QK = tf.matmul(Q, K, transpose_b=True)  # (batch_size, num_heads, seq_len, seq_len)
        QK = QK / tf.math.sqrt(tf.cast(self.depth, tf.float32))
        QK = tf.nn.softmax(QK, axis=-1)
        QK = self.dropout(QK, training=training)

        # Weighted sum of values
        attention_output = tf.matmul(QK, V)  # (batch_size, num_heads, seq_len, depth)

        # Transpose and reshape back to (batch_size, seq_len, output_dim)
        attention_output = tf.transpose(attention_output, perm=[0, 2, 1, 3])
        attention_output = tf.reshape(attention_output, (batch_size, -1, self.output_dim))
        attention_output = self.dropout(attention_output, training=training)

        return attention_output

    def get_config(self):
        config = {
            'output_dim': self.output_dim,
            'num_heads': self.num_heads,
            'dropout_rate': self.dropout_rate,
            'kernel_regularizer': regularizers.serialize(self.kernel_regularizer),
        }
        base_config = super(MultiHeadSelfAttention, self).get_config()
        return dict(list(base_config.items()) + list(config.items()))

    def compute_output_shape(self, input_shape):
        return (input_shape[0], input_shape[1], self.output_dim)
 

In [ ]:
def create_model(mRNA_data,miRNA_data):
    

    
    S_inputs_mRNA = Input(shape=(mRNA_data.shape[1],), dtype='float32',name= 'mRNA_inputs')
  
    S_inputs_miRNA = Input(shape=(miRNA_data.shape[1],), dtype='float32',name= 'miRNA_inputs')
    

   
    h0_mRNA = Biological_module(gene_pathway_bp_dfs[0].shape[0],mapp =gene_pathway_bp_dfs[0].values.T, name = 'h0_mRNA',W_regularizer=l2(0.001))(S_inputs_mRNA)

    
    h0_miRNA = Biological_module(gene_pathway_bp_dfs[1].shape[0],mapp =gene_pathway_bp_dfs[1].values.T, name = 'h0_miRNA',W_regularizer=l2(0.001))(S_inputs_miRNA)



 
    atten1 = MultiHeadSelfAttention(output_dim=128, num_heads=16, W_regularizer=l2(0.005))(h0_mRNA)
    atten2 = MultiHeadSelfAttention(output_dim=128, num_heads=16, W_regularizer=l2(0.005))(h0_miRNA)
    feature_tal = tf.keras.layers.concatenate([atten1,atten2])
  
    
    h2 = tf.keras.layers.Dense(64, activation='tanh')(feature_tal)
    h3 = tf.keras.layers.Dense(32, activation='tanh')(h2)
    h4 = tf.keras.layers.Dense(8, activation='tanh')(h3)
    h5 = tf.keras.layers.Dense(1, activation='sigmoid')(h4)
    
    h5 = tf.squeeze(h5, axis=-2)
    model = Model(inputs=[S_inputs_mRNA,S_inputs_miRNA], outputs=h5)

    model.summary()

    opt = tf.keras.optimizers.Adam(learning_rate = 0.0001,decay=0.0001) 
    model.compile(optimizer=opt,loss='binary_crossentropy',metrics=['acc'])
    return model

In [ ]:
#Evaluation function
import sklearn
from sklearn import metrics
from sklearn.metrics import accuracy_score
from sklearn.metrics import confusion_matrix
from sklearn.metrics import average_precision_score
   

from sklearn.metrics import precision_recall_curve
def get_metrics(true_score,pre_score,pre_probe):
    
  
    fpr, tpr, thresholds = metrics.roc_curve(true_score, pre_probe, pos_label=1)
   
    auc = metrics.auc(fpr, tpr)
    
    aupr = average_precision_score(true_score, pre_probe)
    
    pre, rec, thresholds = precision_recall_curve(true_score, pre_probe)    
    auprc  = metrics.auc(rec, pre)
    
    
    accuracy = accuracy_score(true_score,pre_score)
    
    f1 = metrics.f1_score(true_score, pre_score)
    
    precision = metrics.precision_score(true_score,pre_score)
    
    recall = metrics.recall_score(true_score,pre_score)
    
     
    print( print(confusion_matrix(true_score,pre_score)))
    return precision,accuracy,recall,f1,auc,aupr,auprc


def evaluates(y_test, y_pred):
    
    auc = metrics.roc_auc_score(y_test,y_pred)
    
    aupr = average_precision_score(y_test, y_pred)
    
    precision, recall, thresholds = precision_recall_curve(y_test, y_pred)    
    auprc  = metrics.auc(recall, precision)
    
    pp = [1 if index>=0.5  else 0 for index in  y_pred ]
    
    pre = metrics.precision_score(y_test,pp)
    
    f1 = metrics.f1_score(y_test,pp)
    
    rec = metrics.recall_score(y_test,pp)
    
    acc = metrics.accuracy_score(y_test,pp)
    
    print(confusion_matrix(y_test,pp))
    return pre,acc,rec,f1,auc,aupr,auprc

In [ ]:
y = example_case['response'].values
n_samples =example_case['response'].values

print(len(n_samples),n_samples.sum(),(len(n_samples) -n_samples.sum()))

x_0 =  len(n_samples) / (2*  (len(n_samples) -n_samples.sum()))
x_1 =  len(n_samples) / (2*  n_samples.sum())

print(x_0,x_1)

In [ ]:
#Five-fold cross validation
from sklearn.model_selection import StratifiedKFold
from sklearn.preprocessing import StandardScaler
skf = StratifiedKFold(n_splits=5,shuffle=True,random_state=1030) 

kfscore = []
p = 0
for train_index, test_index in skf.split(mRNA_data.values,y):


    mRNA_train_x = mRNA_data.values[train_index]
    mRNA_test_x  = mRNA_data.values[test_index]

    miRNA_train_x = miRNA_data.values[train_index]
    miRNA_test_x  = miRNA_data.values[test_index]

    #scaler = StandardScaler()
    #mRNA_train_x = scaler.fit_transform(mRNA_train_x)
    #miRNA_train_x = scaler.fit_transform(miRNA_train_x)
    #mRNA_test_x = scaler.fit_transform(mRNA_test_x)
    #miRNA_test_x = scaler.fit_transform(miRNA_test_x)
    train_y  = y[train_index]
    test_y   = y[test_index]
    train_y = np.squeeze(train_y)
    train_y = train_y.astype(int)
    
    model = create_model(mRNA_data,miRNA_data)
    model.fit( { "mRNA_inputs": mRNA_train_x, 'miRNA_inputs':miRNA_train_x},train_y,
                 epochs=200,batch_size = 128,class_weight = {0:x_0,1:x_1})  

    y_pred = model.predict({"mRNA_inputs": mRNA_test_x,'miRNA_inputs':miRNA_test_x})

    y_score = [1 if index>=0.5  else 0 for index in  y_pred]

    evaluate_epoch = get_metrics(test_y,y_score,y_pred)
    print(evaluate_epoch)

    kfscore.append(evaluate_epoch)
    
results = list(np.array(kfscore).sum(axis= 0)/5.0)
print('Cross validated results :  ACC = {}, REC = {}, F1 = {}, AUC = {}, AUPR ={}'.format(results[1],results[2],results[3],results[4],results[5]))